In [ ]:
import os
import gc
import numpy as np
import cv2 as cv
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import LearningRateScheduler
#pylint:disable=no-member (Removes linting problems with cv)

In [ ]:
IMG_SIZE = (80, 80)
channels = 3 # use 1 for grayscale, use 3 for color
char_path = r"../resources/simpsons-characters-data/simpsons_dataset"  # Update this to match your local folder path

In [ ]:
char_dict = {}
for char in os.listdir(char_path):
    full_path = os.path.join(char_path, char)
    if os.path.isdir(full_path):
        char_dict[char] = len(os.listdir(full_path))
char_dict = dict(sorted(char_dict.items(), key=lambda item: item[1], reverse=True))
char_dict

In [ ]:
# Getting the first 10 categories with the most number of images
characters = []
count = 0
for i in char_dict:
    characters.append(i)
    count += 1
    if count >= 10:
        break
characters

In [ ]:
# Create the training and validation datasets
BATCH_SIZE = 32
color_mode = "grayscale" if channels == 1 else "rgb"

train_ds = tf.keras.utils.image_dataset_from_directory(
    char_path,
    class_names=characters,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=color_mode,
    shuffle=True,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    char_path,
    class_names=characters,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=color_mode,
    shuffle=False,
)

In [ ]:
# Number of training and validation samples
print("Training samples:", len(train_ds.file_paths))
print("Validation samples:", len(val_ds.file_paths))

In [ ]:
# Visualizing the data (OpenCV doesn't display well in Jupyter notebooks)
import matplotlib.pyplot as plt

images, labels = next(iter(train_ds))
plt.figure(figsize=(6, 6))
plt.imshow(images[0].numpy().squeeze(), cmap="gray")
plt.title(characters[labels[0]])
plt.axis("off")
plt.show()

In [ ]:
# Useful variables when training
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
# Data augmentation (introduces randomness in network ==> better accuracy)
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Rescaling, Dense, Flatten, Dropout, Conv2D, MaxPooling2D

output_dim = len(characters)
w, h = IMG_SIZE[:2]

model = Sequential([
    Input(shape=(w, h, channels)),
    data_augmentation,
    Rescaling(1.0 / 255),

    Conv2D(32, (3, 3), activation="relu", padding="same"),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.2),

    Conv2D(64, (3, 3), padding="same", activation="relu"),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.2),

    Conv2D(256, (3, 3), padding="same", activation="relu"),
    Conv2D(256, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.2),

    Flatten(),
    Dropout(0.5),
    Dense(1024, activation="relu"),

    # Output Layer
    Dense(output_dim, activation="softmax"),
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

In [ ]:
# Training the model
from tensorflow.keras.callbacks import LearningRateScheduler

def lr_schedule(epoch):
    lr = 0.001
    if epoch >= 8:
        lr *= 0.1
    elif epoch >= 5:
        lr *= 0.5
    return lr

callbacks_list = [LearningRateScheduler(lr_schedule)]

training = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
)

In [ ]:
characters

## Testing

In [ ]:
# provide an image per the characters list above
test_path = r"../resources/simpsons-characters-data/kaggle_simpson_testset/principal/principal_skinner_0.jpg"  # Update to your local path

img = cv.imread(test_path)

plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
# test grayscale 
# ensure channels = 1 above
# def prepare(image):
#     image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
#     image = cv.resize(image, IMG_SIZE)
#     image = np.reshape(image, (1, *IMG_SIZE, 1))
#     return image

# test color
# ensure channels = 3 above
def prepare(image):
    image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
    image = cv.resize(image, IMG_SIZE)
    image = np.reshape(image, (1, *IMG_SIZE, 3))
    return image

In [ ]:
predictions = model.predict(prepare(img))
predicted_index = np.argmax(predictions[0])
confidence = np.max(predictions[0]) * 100

In [ ]:
# Getting class with the highest probability
print(f"{characters[predicted_index]} ({confidence:.2f}%)")

In [ ]:
import json

# 1. Save the trained model weights and architecture
model.save("../resources/simpsons-characters-data/model/simpsons_classifier.keras")

# 2. Save the exact character class order so predictions always map to the right index
with open("../resources/simpsons-characters-data/model/characters.json", "w") as f:
    json.dump(characters, f)